# F2：从相机到空间，再把未来交给规划器

我们先把深度图反投影成三维点，再落到俯视 Occupancy。随后在一个一维连续世界里用 CEM 搜动作，最后观察 Symlog 与梯度裁剪各自解决什么数值问题。

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from hwm.foundations import (
    cem_plan_1d, clip_by_norm, depth_to_points,
    points_to_occupancy, symlog, symexp,
)
print('环境检查通过。')

## 1. 一张深度图怎样变成三维点

深度图中的每个数表示该像素离相机多远。相机内参给出焦距与成像中心。

In [ ]:
depth = np.array([
    [3.0, 3.0, 3.0, 3.0, 3.0],
    [3.0, 2.0, 2.0, 2.0, 3.0],
    [3.0, 2.0, 1.5, 2.0, 3.0],
    [3.0, 2.0, 2.0, 2.0, 3.0],
    [3.0, 3.0, 3.0, 3.0, 3.0],
])
points = depth_to_points(depth, fx=4.0, fy=4.0, cx=2.0, cy=2.0)
print('depth shape:', depth.shape)
print('points shape:', points.shape)
print('中心像素对应点:', points[12])
assert np.allclose(points[12], [0.0, 0.0, 1.5])

只要相机移动，点还要通过外参变换到统一世界坐标。这里先固定相机，只检查反投影。

## 2. 三维点怎样变成俯视 Occupancy

In [ ]:
occupancy = points_to_occupancy(
    points, x_range=(-2, 2), z_range=(0, 4), resolution=0.5
)
print('Occupancy shape:', occupancy.shape)
print('1 表示至少有一个点落入该格：')
print(occupancy)
assert occupancy.sum() > 0

Occupancy 方便回答哪里被占用，却没有保存原始纹理。NeRF 或 3DGS 更适合新视角渲染。表示选择取决于谁使用输出。

## 3. 连续动作太多时怎样搜索

一维小车从 0 出发，5 步后希望到达 3。每步动作是 `[-1,1]` 中任意实数，无法全部枚举。CEM 反复采样、保留较好序列、缩小分布。

In [ ]:
actions, best_scores = cem_plan_1d(
    start=0.0, target=3.0, horizon=5, population=400, elite=40, rounds=5
)
print('动作序列:', np.round(actions, 3))
print('最后位置:', round(float(actions.sum()), 3))
print('每轮最好分数:', [round(x, 3) for x in best_scores])
assert abs(actions.sum() - 3.0) < 0.25

CEM 给出一段临时计划。它没有学出可直接执行的 Policy。PlaNet 每次使用这种搜索，Dreamer 则在想象中训练 Actor。

## 4. 数值跨度太大时会发生什么

奖励可能从 `-100` 到 `100000`。Symlog 保留正负号，同时压缩绝对值。

In [ ]:
values = np.array([-100.0, -1.0, 0.0, 1.0, 100000.0])
encoded = symlog(values)
decoded = symexp(encoded)
for raw, compact in zip(values, encoded):
    print(f'{raw:>10.1f} -> {compact:>8.3f}')
print('往返最大误差:', float(np.max(np.abs(decoded - values))))
assert np.allclose(decoded, values, rtol=1e-5)

Symlog 改变目标尺度。梯度裁剪处理的是另一个问题：一次更新的梯度过大。

In [ ]:
gradient = np.array([30.0, 40.0])
clipped = clip_by_norm(gradient, max_norm=5.0)
print('原梯度 norm:', np.linalg.norm(gradient))
print('裁剪后 norm:', np.linalg.norm(clipped))
print('方向余弦:', np.dot(gradient, clipped) / (
    np.linalg.norm(gradient) * np.linalg.norm(clipped)
))
assert np.isclose(np.linalg.norm(clipped), 5.0)

## 小结与作业

- [ ] 我能说明相机内参如何把像素和三维点联系起来。
- [ ] 我知道 Occupancy 保留结构、舍弃纹理。
- [ ] 我能说明 CEM 是搜索器，不是 Policy。
- [ ] 我能区分目标变换与梯度裁剪。

作业：把相机焦距改为 2 或 8，观察点云怎样变化；把 CEM target 改为 -2，检查动作方向；构造一个错误数据例子，说明梯度裁剪为什么无法修复它。